In [1]:
# ============================================
# Cell 1 — Install required libraries (run once)
# ============================================
# Installs:
# - groq: Groq client for LLM calls
# - sentence-transformers: local embedding models
# - faiss-cpu: dense vector index
# - PyMuPDF (fitz): to extract text from PDFs
# - tqdm: progress bars
!pip install -q groq sentence-transformers faiss-cpu PyMuPDF tqdm

print("Installed dependencies (or they were already installed).")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 40.8 MB/s eta 0:00:00
Installed dependencies (or they were already installed).


In [2]:
# ==================================
# Cell 2 — Enter GROQ API key & client
# ==================================
# Uses getpass to hide your key. The client will be used to call Groq LLM.

from getpass import getpass
import os
from groq import Groq

# Prompt for the API key (hidden). Paste your GROQ API key here.
os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ API key (hidden): ")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("✅ Groq client initialized (llama-3.3-70b-versatile).")


Paste your GROQ API key (hidden): ··········
✅ Groq client initialized (llama-3.3-70b-versatile).


In [3]:
# ==========================================================
# Cell 3 — Imports, local paths, and small helper utilities
# ==========================================================
# This cell defines persistent file locations and small helpers like sha1 for caching.
# You do NOT need to change these unless you want different filenames/locations.

import os
import json
from pathlib import Path
import hashlib
from typing import List, Dict, Tuple
from tqdm import tqdm
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import fitz  # PyMuPDF for reading PDFs (import may warn)

# Directory to save uploaded files inside the Colab VM. Created if missing.
DATA_DIR = Path("uploaded_docs")
DATA_DIR.mkdir(exist_ok=True)

# Files used to persist state (embedding cache, metadata, faiss index)
EMBED_CACHE_PATH = Path("embed_cache.json")    # map sha1(text) -> embedding list
METADATA_PATH = Path("doc_metadata.json")      # store chunk ids and texts
FAISS_INDEX_PATH = Path("faiss.index")         # binary FAISS index

def sha1(text: str) -> str:
    """
    Return a SHA-1 hex digest for `text`. Used as a cache key for embeddings.
    """
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

print("Utility setup done. Paths:")
print(" - uploaded files dir:", DATA_DIR.resolve())
print(" - embed cache path:", EMBED_CACHE_PATH.resolve())
print(" - metadata path:", METADATA_PATH.resolve())
print(" - faiss index path:", FAISS_INDEX_PATH.resolve())


Utility setup done. Paths:
 - uploaded files dir: /content/uploaded_docs
 - embed cache path: /content/embed_cache.json
 - metadata path: /content/doc_metadata.json
 - faiss index path: /content/faiss.index


In [4]:
# =========================================================
# 📄 Dynamic Chunking Function (Auto-Adjusting Version)
# =========================================================
# This function splits long text into overlapping "chunks"
# that are small enough to fit inside a model's context window.
#
# Why chunking is important:
#  - LLMs (like Groq's models) have input size limits.
#  - We must split long documents into smaller parts ("chunks").
#  - Adding small overlap helps the model maintain context continuity
#    between consecutive chunks.
#
# This version is *dynamic* — it adjusts chunk size based on total text length.
# Small files = fewer, larger chunks; big files = more, smaller chunks.

from typing import List

def dynamic_chunk_text(text: str, base_tokens: int = 300, overlap_ratio: float = 0.15) -> List[str]:
    """
    Dynamically split input text into overlapping chunks.

    Args:
        text (str): The complete text to be divided into chunks.
        base_tokens (int): Average number of words per chunk.
                           The function will adapt this dynamically.
        overlap_ratio (float): Fraction (0–1) of overlap between chunks.
                               Example: 0.15 = 15% overlap.

    Returns:
        List[str]: A list of overlapping text chunks.

    Working logic:
        1. Count total words in the text.
        2. Decide chunk size based on document length.
        3. Add some overlap to preserve context between chunks.
        4. Return a list of chunk strings.
    """

    # --- 1️⃣ Handle empty text case ---
    if not text or not text.strip():
        # If text is empty or whitespace, return nothing
        return []

    # --- 2️⃣ Split text into words ---
    # Using a simple word split (good enough for most text data)
    words = text.split()
    total_words = len(words)

    # --- 3️⃣ Determine chunk size dynamically ---
    # We automatically adjust based on how long the document is
    if total_words <= 500:
        # Very small doc → just one chunk
        target_tokens = total_words
    elif total_words <= 3000:
        # Medium doc → normal chunk size (base_tokens)
        target_tokens = base_tokens
    else:
        # Large doc → slightly smaller chunks for memory efficiency
        # Example: 10,000 words → about 90–150 words per chunk
        target_tokens = max(200, int(base_tokens * (3000 / total_words)))

    # --- 4️⃣ Compute overlap size (in words) ---
    # Overlap keeps some words repeated at boundaries so context flows naturally
    overlap = int(target_tokens * overlap_ratio)

    # --- 5️⃣ Chunk generation loop ---
    chunks = []  # stores resulting chunks
    i = 0        # start index of current chunk

    while i < total_words:
        # Slice words into a chunk
        chunk = words[i:i + target_tokens]
        chunks.append(" ".join(chunk))  # join back into a string

        # Move window ahead, leaving some overlap for the next chunk
        step = max(1, target_tokens - overlap)
        i += step  # shift window by (chunk size - overlap)

    # --- 6️⃣ Return the final list of chunks ---
    return chunks


# --- 🔍 Quick Demo to Test Function ---
# Here we simulate a 5,000-word text to verify how chunks are formed.
example_text = " ".join([f"word{i}" for i in range(1, 5001)])  # fake 5000-word doc
sample_chunks = dynamic_chunk_text(example_text)

print(f"✅ Dynamic chunking complete — total chunks: {len(sample_chunks)}")
print(f"Example chunk length (words): {len(sample_chunks[0].split())}")

✅ Dynamic chunking complete — total chunks: 30
Example chunk length (words): 200


In [5]:
# ============================================================
# Cell 5 — Upload helper and file text extraction functions
# ============================================================
# This cell provides:
# - load_uploaded_files(): opens a Colab upload dialog and writes files to DATA_DIR
# - extract_text_from_pdf(): extracts text from PDFs using PyMuPDF (fitz)
#
# Supported file types: .txt, .md, .csv, .pdf
# For PDFs that are scanned images (no text layer), OCR is required (not included).

from google.colab import files

def extract_text_from_pdf(path: Path) -> str:
    """
    Read a PDF from `path` and return extracted text from every page.
    Uses PyMuPDF (fitz). Returns concatenated text.
    """
    text_parts: List[str] = []
    # Open the PDF file
    with fitz.open(str(path)) as doc:
        # Iterate pages and extract text (get_text returns page text)
        for page in doc:
            text_parts.append(page.get_text())
    # Join all page texts with newlines and return
    return "\n".join(text_parts)

def load_uploaded_files() -> List[Tuple[str, str]]:
    """
    Launches a file upload dialog in Colab.
    Saves uploaded bytes to DATA_DIR and returns list of (path, text) tuples.

    Returns:
      loaded: list of tuples (path_str, extracted_text)
    """
    # This opens an upload dialog — choose files from your local machine.
    uploaded = files.upload()
    loaded: List[Tuple[str, str]] = []
    # Iterate uploaded files
    for fname, content in uploaded.items():
        out_path = DATA_DIR / fname
        # Write raw bytes to file in uploaded_docs/
        out_path.write_bytes(content)
        # Determine how to read the file based on extension
        if fname.lower().endswith(".pdf"):
            # Extract text from PDF
            text = extract_text_from_pdf(out_path)
        else:
            # Assume text-like file; read text with utf-8 and ignore errors
            text = out_path.read_text(encoding="utf-8", errors="ignore")
        loaded.append((str(out_path), text))
    # Return list of (path, text)
    return loaded

print("Upload and file-reading helpers ready. Use load_uploaded_files() to upload files.")


Upload and file-reading helpers ready. Use load_uploaded_files() to upload files.


In [6]:
# ===============================================================
# Cell 6 — Setup embedder, embedding cache, metadata, and FAISS
# ===============================================================
# - Loads a SentenceTransformer embedder
# - Loads embed_cache.json if exists (to avoid recomputing embeddings)
# - Loads metadata.json (ids and texts) if exists
# - Loads FAISS index from disk if present (so persistence works across sessions)

EMBED_MODEL = "all-MiniLM-L6-v2"  # lightweight and effective for RAG
embedder = SentenceTransformer(EMBED_MODEL)  # load model (may take a few seconds)

def load_embed_cache(path: Path) -> Dict[str, List[float]]:
    """
    Load embedding cache from JSON. The cache maps sha1(chunk_text) -> embedding list.
    If file missing or unreadable, returns empty dict.
    """
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_embed_cache(path: Path, cache: Dict[str, List[float]]):
    """Save the embedding cache to JSON (atomic write would be better for production)."""
    path.write_text(json.dumps(cache), encoding="utf-8")

def load_metadata() -> Dict:
    """
    Load metadata file containing:
      - 'ids': list of chunk ids (one per chunk)
      - 'texts': list of chunk texts, same order as ids
    Returns a dict with those keys if file exists; else default empty lists.
    """
    if METADATA_PATH.exists():
        try:
            return json.loads(METADATA_PATH.read_text(encoding="utf-8"))
        except Exception:
            return {"ids": [], "texts": []}
    return {"ids": [], "texts": []}

def save_metadata(meta: Dict):
    """Persist metadata dict to a JSON file."""
    METADATA_PATH.write_text(json.dumps(meta), encoding="utf-8")

def load_faiss_index_if_exists() -> faiss.IndexFlat:
    """
    If a FAISS index file exists at FAISS_INDEX_PATH, read it and return.
    Otherwise return None (no index yet).
    """
    if FAISS_INDEX_PATH.exists():
        return faiss.read_index(str(FAISS_INDEX_PATH))
    return None

# Load cache and metadata at startup
embed_cache = load_embed_cache(EMBED_CACHE_PATH)
metadata = load_metadata()
faiss_index = load_faiss_index_if_exists()

print("Embedder loaded and persistence helpers ready.")
print(f" - cached embeddings: {len(embed_cache)}")
print(f" - stored chunks: {len(metadata.get('texts', []))}")
print(f" - faiss index loaded: {faiss_index is not None}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded and persistence helpers ready.
 - cached embeddings: 0
 - stored chunks: 0
 - faiss index loaded: False


In [7]:
# ========================================================
# Cell 7 — Add uploaded docs to FAISS index (core function)
# ========================================================
# This function:
# 1) takes list of (path, text)
# 2) chunks each document
# 3) computes embeddings (with caching)
# 4) appends embeddings to FAISS index (or creates it)
# 5) updates metadata and persists everything

def add_documents_to_index(docs: List[Tuple[str, str]],
                           base_tokens: int = 300, # Changed from target_tokens
                           overlap_ratio: float = 0.15, # Changed from overlap and type
                           persist: bool = True):
    """
    Add a list of (source_path, text) documents to the FAISS index.

    Steps:
      - chunk each document using dynamic_chunk_text
      - for each chunk: compute sha1, check cache, embed if missing
      - collect embeddings and chunk ids/texts
      - normalize embeddings and add to FAISS (create or append)
      - persist embed cache, metadata, and FAISS index files
    """
    global embed_cache, metadata, faiss_index

    # Step A: build lists of chunk texts and chunk IDs from incoming docs
    new_texts: List[str] = []
    new_ids: List[str] = []

    for src_path, text in docs:
        # Derive a readable base name for chunk IDs (avoid long paths)
        base = Path(src_path).name
        # Split the document text into smaller chunks
        chunks = dynamic_chunk_text(text, base_tokens=base_tokens, overlap_ratio=overlap_ratio) # Changed arguments
        # For each chunk, create an ID and add to lists
        for i, chunk in enumerate(chunks):
            # chunk ID format: filename__c{index}
            chunk_id = f"{base}__c{i+1}"
            new_ids.append(chunk_id)
            new_texts.append(chunk)

    if not new_texts:
        # Nothing to add; maybe an empty upload
        print("No chunks found in provided documents (empty text?).")
        return

    # Step B: compute embeddings for all new chunks (with caching)
    emb_arrays = []
    for txt, cid in tqdm(zip(new_texts, new_ids), total=len(new_texts), desc="Embedding chunks"):
        # Cache key is sha1 of chunk text to avoid recomputing embeddings for identical text
        key = sha1(txt)
        if key in embed_cache:
            # If cached, convert stored list back to numpy array
            emb = np.array(embed_cache[key], dtype=np.float32)
        else:
            # Compute new embedding and store it in cache as list for JSON persistence
            emb = embedder.encode(txt, convert_to_numpy=True).astype(np.float32)
            embed_cache[key] = emb.tolist()
        emb_arrays.append(emb)

    # Step C: update metadata lists (ids and texts)
    metadata["ids"].extend(new_ids)
    metadata["texts"].extend(new_texts)

    # Stack embeddings into a single 2D array
    embs = np.vstack(emb_arrays)  # shape: (N_new, dim)
    dim = embs.shape[1]

    # Normalize embeddings for cosine similarity (FAISS IndexFlatIP uses inner product)
    faiss.normalize_L2(embs)

    # Step D: create or append to FAISS index
    if faiss_index is None:
        # No index yet: create new IndexFlatIP with appropriate dimension and add embeddings
        faiss_index = faiss.IndexFlatIP(dim)
        faiss_index.add(embs)
    else:
        # Index exists: verify dimension matches and append
        if faiss_index.d != dim:
            raise ValueError(f"Existing FAISS dim={faiss_index.d} does not match new emb dim={dim}")
        faiss_index.add(embs)

    # Step E: persist embed_cache, metadata, and FAISS index to disk if requested
    if persist:
        save_embed_cache(EMBED_CACHE_PATH, embed_cache)
        save_metadata(metadata)
        faiss.write_index(faiss_index, str(FAISS_INDEX_PATH))
        print(f"Persisted embed cache ({len(embed_cache)}), metadata, and FAISS index.")

    print(f"Added {len(new_texts)} chunks to index. Total vectors now: {faiss_index.ntotal}.")

In [8]:
# =====================================================
# Cell 8 — Retrieval + build prompt + call Groq (RAG)
# =====================================================
# This cell contains:
# - retrieve(): query FAISS and return top-k chunks + scores
# - build_prompt_from_retrieved(): create a system+user messages list for Groq
# - rag_query(): full pipeline to retrieve, prompt, and call Groq

def retrieve(query: str, top_k: int = 4) -> List[Dict]:
    """
    Retrieve top_k most similar chunks to the query using FAISS.
    Returns list of dicts: {id, text, score}
    """
    # If index missing or empty, nothing to retrieve
    if faiss_index is None or faiss_index.ntotal == 0:
        return []

    # Step 1: embed the query using the same embedder used for documents
    qvec = embedder.encode([query], convert_to_numpy=True).astype(np.float32)
    # Step 2: normalize query vector for cosine similarity
    faiss.normalize_L2(qvec)
    # Step 3: perform FAISS search (returns distances and indices)
    D, I = faiss_index.search(qvec, top_k)  # D shape: (1, top_k); I shape: (1, top_k)

    # Build results list
    results: List[Dict] = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        results.append({
            "id": metadata["ids"][idx],
            "text": metadata["texts"][idx],
            "score": float(score)
        })
    return results

def build_prompt_from_retrieved(query: str, retrieved: List[Dict], allow_fallback: bool = True, max_context_chars: int = 3000):
    """
    Construct a system+user messages list for the Groq LLM.
      - If allow_fallback is True, the system message allows the model to use general knowledge
        when the context is insufficient.
      - The function concatenates retrieved chunk texts into a context string up to max_context_chars.
    Returns:
      messages: list[dict] suitable for client.chat.completions.create(...)
    """
    ctx_parts = []
    total = 0
    # Add retrieved chunks one by one until reaching the char budget
    for r in retrieved:
        part = f"[{r['id']}]\n{r['text']}"
        if total + len(part) <= max_context_chars:
            ctx_parts.append(part)
            total += len(part)
        else:
            # If there's remaining room, add a truncated piece to not exceed budget
            remain = max_context_chars - total
            if remain > 50:
                ctx_parts.append(part[:remain] + "...")
                total += remain
            break

    # If no context parts were added, indicate that explicitly
    context_text = "\n\n---\n\n".join(ctx_parts) if ctx_parts else "NO_CONTEXT_AVAILABLE"

    if allow_fallback:
        system_msg = (
            "You are a helpful assistant. Prefer the provided context when answering. "
            "If the context is insufficient, you may use your general knowledge but please indicate you did so."
        )
    else:
        system_msg = (
            "You are a helpful assistant. ANSWER ONLY using the provided context. "
            "If the answer is not present, reply: 'I don't know based on the provided documents.'"
        )

    user_msg = f"Context:\n{context_text}\n\nUser question: {query}\n\nPlease answer succinctly and, if using context, cite the relevant doc ids like [file__c1]."

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]

def rag_query(query: str, top_k: int = 4, allow_fallback: bool = True, model: str = "llama-3.3-70b-versatile"):
    """
    High-level RAG function:
      - retrieve top_k chunks
      - build prompt messages
      - call Groq LLM and return answer + retrieved chunks
    """
    # 1) retrieve
    retrieved = retrieve(query, top_k=top_k)
    # 2) build prompt messages for Groq
    messages = build_prompt_from_retrieved(query, retrieved, allow_fallback=allow_fallback)
    # 3) call the Groq model
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=400,
            temperature=0.0,  # low temperature -> deterministic answers
        )
        answer = resp.choices[0].message.content.strip()
    except Exception as e:
        # Return an error string if the API call fails
        answer = f"[Groq API error: {e}]"

    return {"answer": answer, "retrieved": retrieved}


In [9]:
# =====================================================
# Cell 9 — Interactive driver: upload, add, query, info
# =====================================================
# This cell presents a simple command-driven interface:
# Commands:
#  - upload : open file dialog and upload files (saved to uploaded_docs/)
#  - add    : add files from uploaded_docs/ that are not yet indexed
#  - query  : ask a question (runs rag_query)
#  - info   : show index and cache status
#  - exit   : quit interactive loop
#
# Typical flow:
# 1) run cell
# 2) type 'upload' to upload files, then 'add' or 'upload' with add now=yes
# 3) type 'query' to ask questions

print("RAG Chat Driver ready. Commands: upload, add, query, info, exit\n")

while True:
    cmd = input("Command (upload/add/query/info/exit): ").strip().lower()
    if cmd in ("exit", "quit"):
        print("Exiting. Bye 👋")
        break

    if cmd == "upload":
        # Upload files via Colab widget
        print("Upload files (.txt, .md, .pdf). They will be saved into 'uploaded_docs/'.")
        loaded = load_uploaded_files()  # returns list of (path, text)
        print(f"Uploaded {len(loaded)} files.")
        # Ask whether to add them immediately to the index
        add_now = input("Add uploaded files to index now? (y/N): ").strip().lower()
        if add_now == "y":
            add_documents_to_index(loaded)
        else:
            print("Files saved in uploaded_docs/. Run 'add' later to index them.")

    elif cmd == "add":
        # Add any files in uploaded_docs that are not yet in metadata
        files_in_dir = sorted(DATA_DIR.glob("*"))
        docs_to_add: List[Tuple[str, str]] = []
        for p in files_in_dir:
            # Use file name presence in metadata ids to detect if we've indexed it
            already_indexed = any(str(p.name) in mid for mid in metadata.get("ids", []))
            if not already_indexed:
                # read file text (pdf or text)
                if p.suffix.lower() == ".pdf":
                    txt = extract_text_from_pdf(p)
                else:
                    txt = p.read_text(encoding="utf-8", errors="ignore")
                docs_to_add.append((str(p), txt))
        if docs_to_add:
            print(f"Adding {len(docs_to_add)} files to the index...")
            add_documents_to_index(docs_to_add)
        else:
            print("No new files to add. All uploaded files already indexed (or uploaded_docs/ empty).")

    elif cmd == "query":
        # Ask a question and run RAG
        q = input("Enter your question: ").strip()
        if not q:
            print("Empty question — try again.")
            continue
        # Ask user if fallback allowed (default Yes)
        allow_fb_input = input("Allow fallback to general knowledge if context insufficient? (Y/n): ").strip().lower()
        allow_fallback = not (allow_fb_input == "n")
        # Run the RAG pipeline
        out = rag_query(q, top_k=4, allow_fallback=allow_fallback)
        # Show retrieved chunks (if any)
        print("\n--- Retrieved chunks (top results) ---")
        if out["retrieved"]:
            for r in out["retrieved"]:
                print(f"- {r['id']} (score={r['score']:.3f})")
                # print first 200 characters for preview
                print("  ", r['text'][:200].replace("\n", " "), "...")
        else:
            print("- No context: FAISS index empty or no matches found.")
        # Show final model answer
        print("\n--- Final Answer ---")
        print(out["answer"])
        print("\n" + "="*80 + "\n")

    elif cmd == "info":
        # Show helpful runtime info
        total_vectors = faiss_index.ntotal if faiss_index is not None else 0
        print(f"Index vectors: {total_vectors}")
        print(f"Stored chunks: {len(metadata.get('texts', []))}")
        print(f"Cached embeddings: {len(embed_cache)}")
        print(f"Uploaded files folder: {DATA_DIR.resolve()}")
    else:
        print("Unknown command. Use one of: upload, add, query, info, exit")


RAG Chat Driver ready. Commands: upload, add, query, info, exit

Command (upload/add/query/info/exit): upload
Upload files (.txt, .md, .pdf). They will be saved into 'uploaded_docs/'.


Saving major project 1pdf.pdf to major project 1pdf.pdf
Uploaded 1 files.
Add uploaded files to index now? (y/N): y


Embedding chunks: 100%|██████████| 19/19 [00:03<00:00,  6.10it/s]


Persisted embed cache (19), metadata, and FAISS index.
Added 19 chunks to index. Total vectors now: 19.
Command (upload/add/query/info/exit): y
Unknown command. Use one of: upload, add, query, info, exit
Command (upload/add/query/info/exit): y
Unknown command. Use one of: upload, add, query, info, exit
Command (upload/add/query/info/exit): info
Index vectors: 19
Stored chunks: 19
Cached embeddings: 19
Uploaded files folder: /content/uploaded_docs
Command (upload/add/query/info/exit): query
Enter your question: what actully in the pdf what it about
Allow fallback to general knowledge if context insufficient? (Y/n): y

--- Retrieved chunks (top results) ---
- major project 1pdf.pdf__c2 (score=0.160)
   Internal Assessment have been incorporated in the report deposited in the department liberary. The project report has been approved as it satisfies the academic requirements in respect to project work ...
- major project 1pdf.pdf__c3 (score=0.085)
   for their support and guidance. Also we













*  Type upload → choose .txt / .pdf files in the dialog.

* Choose whether to add uploaded files to index now. If you say yes, indexing runs immediately.

*  Or type add later to add files already in uploaded_docs/.

* Type query → enter a natural-language question. You’ll see retrieved chunks (top results) and the Groq-generated answer.

*   Type info to see status.

*   Type exit to quit.



